# 1.0 · First look at MSLesSeg

Goal: load one study, check shapes/orientation and visualise FLAIR, T1, T2 with the lesion mask.

Related issues: #2, #4

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from ms_seg.config import RAW_DATA_DIR

DATA = RAW_DATA_DIR / "MSLesSeg"
print(DATA, DATA.exists())

## Dataset layout

- `train/P<id>/T<timepoint>/P<id>_T<tp>_{FLAIR,T1,T2,MASK}.nii.gz` (53 patients, 1–4 timepoints)
- `test/P<id>/P<id>_{FLAIR,T1,T2,MASK}.nii.gz` (22 patients, 1 timepoint)
- `info_dataset/` clinical data and scanner info

In [ ]:
train_studies = sorted(DATA.glob("train/P*/T*"))
test_studies = sorted(DATA.glob("test/P*"))
print(f"train patients: {len(list(DATA.glob('train/P*')))} | train studies: {len(train_studies)}")
print(f"test patients:  {len(test_studies)}")

In [ ]:
def load_study(study_dir):
    """Return a dict with FLAIR, T1, T2 and MASK arrays for one study folder."""
    out = {}
    for mod in ["FLAIR", "T1", "T2", "MASK"]:
        f = next(study_dir.glob(f"*_{mod}.nii.gz"))
        img = nib.load(f)
        out[mod] = img.get_fdata(dtype=np.float32)
    out["affine"] = img.affine
    return out

study = train_studies[0]
s = load_study(study)
print(study.relative_to(DATA))
for k in ["FLAIR", "T1", "T2", "MASK"]:
    print(f"{k:5s} shape={s[k].shape} min={s[k].min():.1f} max={s[k].max():.1f}")
print("orientation:", nib.aff2axcodes(s['affine']))
print("lesion volume (voxels = mm³):", int(s['MASK'].sum()))

## Axial slice with the most lesion

In [ ]:
z = int(np.argmax(s["MASK"].sum(axis=(0, 1))))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, mod in zip(axes, ["FLAIR", "T1", "T2"]):
    ax.imshow(s[mod][:, :, z].T, cmap="gray", origin="lower")
    ax.set_title(mod)
axes[3].imshow(s["FLAIR"][:, :, z].T, cmap="gray", origin="lower")
axes[3].imshow(np.ma.masked_where(s["MASK"][:, :, z].T == 0, s["MASK"][:, :, z].T), cmap="autumn", alpha=0.6, origin="lower")
axes[3].set_title("FLAIR + mask")
for ax in axes:
    ax.axis("off")
fig.suptitle(f"{study.relative_to(DATA)} — axial slice z={z}")
plt.tight_layout()

## Three orthogonal views

In [ ]:
m = s["MASK"]
x = int(np.argmax(m.sum(axis=(1, 2))))
y = int(np.argmax(m.sum(axis=(0, 2))))
views = {
    "sagittal": (s["FLAIR"][x, :, :], m[x, :, :]),
    "coronal": (s["FLAIR"][:, y, :], m[:, y, :]),
    "axial": (s["FLAIR"][:, :, z], m[:, :, z]),
}
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (name, (img, msk)) in zip(axes, views.items()):
    ax.imshow(img.T, cmap="gray", origin="lower")
    ax.imshow(np.ma.masked_where(msk.T == 0, msk.T), cmap="autumn", alpha=0.6, origin="lower")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()

## Next steps

- EDA over all studies: lesion volume, number and size of lesions, intensities, scanners (#3)
- Define the 5 cross-validation folds at patient level (#5)